<a href="https://colab.research.google.com/github/EstherSudargo/Banana-Segmentation-using-YOLO26-and-YOLO26-Vmamba/blob/main/image%26label_rotation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install -U albumentations opencv-python-headless

In [9]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import os
import cv2


test_img = "/content/drive/MyDrive/banana_yolo26/train/images/image_0.jpg"
test_txt = test_img.replace("images", "labels").replace(".jpg", ".txt").replace(".png", ".txt")

print(f"Checking label file: {test_txt}")
if os.path.exists(test_txt):
    with open(test_txt, 'r') as f:
        lines = f.readlines()
        print(f"--- RAW FILE CONTENT ---")
        print(lines)
        print(f"------------------------")

    if os.path.exists(test_img):
        img = cv2.imread(test_img)
        h, w, _ = img.shape
        print(f"Image dimensions: Width={w}, Height={h}")
else:
    print(f"Error: Could not find a label file at: {test_txt}")
    print("Make sure your original 'labels' folder is parallel to your 'images' folder.")

Checking label file: /content/drive/MyDrive/banana_yolo26/train/labels/image_0.txt
--- RAW FILE CONTENT ---
['0 0.295400 0.138980 0.299890 0.156920 0.309590 0.178810 0.320810 0.199730 0.337240 0.218680 0.354450 0.238590 0.375370 0.252550 0.398540 0.264500 0.421700 0.270500 0.444820 0.267480 0.461300 0.259510 0.482220 0.245550 0.493390 0.228660 0.500110 0.208700 0.503140 0.192770 0.501630 0.179810 0.498650 0.164890 0.500110 0.146940 0.506840 0.133030 0.500890 0.122040 0.491140 0.115080 0.488170 0.105100 0.482220 0.117050 0.479970 0.126020 0.482220 0.136960 0.482220 0.158890 0.477730 0.173860 0.462760 0.186820 0.449350 0.198760 0.429890 0.204720 0.408970 0.203750 0.388780 0.192770 0.368640 0.177840 0.353670 0.161910 0.338750 0.144980 0.329050 0.133030 0.317830 0.121030 0.306610 0.117050 0.297640 0.121030\n', '0 0.429160 0.400960 0.437350 0.390020 0.448570 0.372080 0.447060 0.351160 0.450810 0.331240 0.462030 0.312340 0.477730 0.299380 0.502360 0.294390 0.536000 0.298370 0.562140 0.313300

In [11]:
import os
import cv2
import numpy as np
import yaml

# 1. Dataset Configuration
data_config = {
    'path': '/content/drive/MyDrive/banana_yolo26',
    'train': 'train/images',
    'val': 'valid/images',
    'names': {
        0: 'banana',
    }
}
with open('/content/rotated_dataset.yaml', 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("SUCCESS: /content/rotated_dataset.yaml has been created!")

SUCCESS: /content/rotated_dataset.yaml has been created!


In [12]:
import os

def sanitize_and_fix_yolo_labels(labels_dir):
    """
    Cleans up original YOLO text label files by removing metadata headers,
    filtering out non-integer class rows, and enforcing even coordinate pairs.
    """
    if not os.path.exists(labels_dir):
        print(f"Directory not found: '{labels_dir}'. Skipping.")
        return

    txt_files = [f for f in os.listdir(labels_dir) if f.lower().endswith('.txt')]
    print(f"Cleaning {len(txt_files)} label files in: {labels_dir}...")

    cleaned_count = 0

    for txt_file in txt_files:
        file_path = os.path.join(labels_dir, txt_file)
        valid_yolo_lines = []
        file_was_modified = False

        with open(file_path, 'r') as f:
            lines = f.readlines()

        for line in lines:
            line_str = line.strip()
            # 1. Drop completely empty lines or lines starting with non-YOLO metadata
            if not line_str: # Handles completely empty lines
                file_was_modified = True
                continue

            try:
                parts = line_str.split()
                if len(parts) < 1: # Ensure there's at least a class_id
                    raise ValueError("Line is too short to be a valid YOLO label.")

                class_id = int(parts[0]) # This can raise ValueError
                coords = [float(x) for x in parts[1:]] # This can raise ValueError

                # 3. Enforce valid coordinate pairs (X, Y) for segmentation
                # Remaining elements must be an even number.
                if len(coords) % 2 != 0:
                    # Odd length means a trailing coordinate is missing its pair. Cut the orphan out.
                    coords = coords[:-1]
                    file_was_modified = True

                # Reconstruct clean row with 6 decimal places for coordinates
                clean_line = f"{class_id} " + " ".join([f"{c:.6f}" for c in coords])
                valid_yolo_lines.append(clean_line)

            except ValueError:
                # If parsing fails (e.g., class_id not int, coords not float, line too short), it's corrupted.
                file_was_modified = True
                continue

        # Only overwrite the file if changes were actually made
        if file_was_modified or len(valid_yolo_lines) != len(lines):
            with open(file_path, 'w') as f:
                # Add a newline at the end of the file if there are lines to write
                if valid_yolo_lines:
                    f.write("\n".join(valid_yolo_lines) + "\n")
                else:
                    f.write("") # Write an empty file if no valid lines
            cleaned_count += 1

    print(f"Successfully sanitized {cleaned_count} files in {labels_dir}.\n")

# --- EXECUTION ---
# Path configurations mapped to your original layout
BASE_PATH = '/content/drive/MyDrive/banana_yolo26'
TRAIN_LABELS = os.path.join(BASE_PATH, 'train/labels')
VALID_LABELS = os.path.join(BASE_PATH, 'valid/labels')

# Clean original folders
sanitize_and_fix_yolo_labels(TRAIN_LABELS)
sanitize_and_fix_yolo_labels(VALID_LABELS)


Cleaning 6 label files in: /content/drive/MyDrive/banana_yolo26/train/labels...
Successfully sanitized 0 files in /content/drive/MyDrive/banana_yolo26/train/labels.

Cleaning 1 label files in: /content/drive/MyDrive/banana_yolo26/valid/labels...
Successfully sanitized 0 files in /content/drive/MyDrive/banana_yolo26/valid/labels.



In [13]:
def rotate_image_and_coordinates(image, coordinates_list, angle):
    """
    Rotates an image and recalculates all normalized YOLO coordinates
    (works flawlessly for both bounding boxes and segmentation polygons).
    """
    h, w = image.shape[:2]
    cx, cy = w / 2, h / 2

    # Calculate rotation matrix
    matrix = cv2.getRotationMatrix2D((cx, cy), angle, 1.0)

    # Calculate new bounding dimensions of the image so nothing gets cut off
    cos = np.abs(matrix[0, 0])
    sin = np.abs(matrix[0, 1])
    new_w = int((h * sin) + (w * cos))
    new_h = int((h * cos) + (w * sin))

    # Adjust translation mapping based on new canvas size
    matrix[0, 2] += (new_w / 2) - cx
    matrix[1, 2] += (new_h / 2) - cy

    # Rotate the image
    rotated_image = cv2.warpAffine(image, matrix, (new_w, new_h), borderMode=cv2.BORDER_CONSTANT, borderValue=(0,0,0))

    # Rotate the label coordinates
    rotated_coordinates_list = []
    for coords in coordinates_list:
        class_id = coords[0]
        pts = np.array(coords[1:]).reshape(-1, 2)

        # Convert normalized coordinates back to absolute pixels relative to original size
        pts[:, 0] *= w
        pts[:, 1] *= h

        # Apply the rotation matrix to the points
        ones = np.ones(shape=(len(pts), 1))
        pts_ones = np.concatenate((pts, ones), axis=1)
        transformed_pts = matrix.dot(pts_ones.T).T

        # Re-normalize the coordinates relative to the NEW image size
        transformed_pts[:, 0] /= new_w
        transformed_pts[:, 1] /= new_h

        # Ensure values stay strictly clamped between 0.0 and 1.0
        transformed_pts = np.clip(transformed_pts, 0.0, 1.0)

        # Flatten back out to YOLO format string structure
        flattened_coords = [class_id] + transformed_pts.flatten().tolist()
        rotated_coordinates_list.append(flattened_coords)

    return rotated_image, rotated_coordinates_list

def process_yolo_dataset(base_path, relative_images_path, angle_step=10):
    images_dir = os.path.join(base_path, relative_images_path)
    labels_dir = images_dir.replace('images', 'labels')

    # Check if the image directory exists
    if not os.path.exists(images_dir):
        print(f"Error: The image directory '{images_dir}' does not exist.")
        print("Please ensure your dataset is correctly organized in Google Drive and that 'images' and 'labels' folders are present.")
        return # Exit the function if the directory doesn't exist

    # Target folders
    output_images_dir = images_dir + f"_rotated_{angle_step}deg_final"
    output_labels_dir = labels_dir + f"_rotated_{angle_step}deg_final"

    os.makedirs(output_images_dir, exist_ok=True)
    os.makedirs(output_labels_dir, exist_ok=True)

    valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp')
    image_files = [f for f in os.listdir(images_dir) if f.lower().endswith(valid_extensions)]

    print(f"Processing {len(image_files)} images from {relative_images_path}...")

    for img_file in image_files:
        base_name = os.path.splitext(img_file)[0]
        img_path = os.path.join(images_dir, img_file)
        lbl_path = os.path.join(labels_dir, base_name + '.txt')

        image = cv2.imread(img_path)
        if image is None:
            continue

        # Parse text coordinates manually
        parsed_labels = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) >= 5:  # Bounding boxes have 5 parts, Polygons have many more
                        class_id = int(parts[0])
                        coords = [float(x) for x in parts[1:]]
                        parsed_labels.append([class_id] + coords)

        # Generate 360-degree permutations based on angle step
        for angle in range(0, 360, angle_step):
            rot_img, rot_labels = rotate_image_and_coordinates(image, parsed_labels, angle)

            new_name = f"{base_name}_rot{angle:03d}"

            # Save Image
            cv2.imwrite(os.path.join(output_images_dir, f"{new_name}.jpg"), rot_img)

            # Save Label Text File
            with open(os.path.join(output_labels_dir, f"{new_name}.txt"), 'w') as f:
                for label in rot_labels:
                    class_id = label[0]
                    coords_str = " ".join([f"{x:.6f}" for x in label[1:]])
                    f.write(f"{class_id} {coords_str}\n")

    print(f"Saved successfully to: {output_images_dir}")

# --- Run the Script ---
# Change angle_step=10 to angle_step=1 if you want every single degree (360 frames per image)
process_yolo_dataset(data_config['path'], data_config['train'], angle_step=1)
process_yolo_dataset(data_config['path'], data_config['val'], angle_step=1)


Processing 6 images from train/images...
Saved successfully to: /content/drive/MyDrive/banana_yolo26/train/images_rotated_1deg_final
Processing 1 images from valid/images...
Saved successfully to: /content/drive/MyDrive/banana_yolo26/valid/images_rotated_1deg_final


In [15]:
import yaml
import os


data_config = {
    'path': '/content/drive/MyDrive/]banana_yolo26',
    'train': 'train/images_rotated_1deg_final',
    'valid': 'valid/images_rotated_1deg_final',
    'names': {
        0: 'banana',
    }
}

with open('/content/custom_dataset.yaml', 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("SUCCESS: /content/custom_dataset.yaml has been created!")

SUCCESS: /content/custom_dataset.yaml has been created!
